# POGGLexiconUtil usage examples

[See full module documentation here.](project:/apidocs/pogg/pogg.lexicon.lexicon_builder.md)

The `POGGLexiconUtil` class contains a number of static functions to make working with POGG lexicons easier. If you're trying to create a lexicon in the POGG format, you won't need to directly interact with a majority of the functions in `POGGLexiconUtil`, as many of them are just helper functions for other functions (e.g. `validate_node_entry`).

This guide gives examples for the functions you would be most likely to work with directly:

- [create_lexicon_skeleton example](#create-lexicon-skeleton-example)
- [initialize_lexicon_directory example](#initialize-lexicon-directory-example)
- [update_lexicon_files example](#update-lexicon-files-example)

For the examples on this page we will need the following imports:

In [20]:
import json
import pprint as pp
from pogg.lexicon.lexicon_builder import POGGLexiconUtil

## `create_lexicon_skeleton` example

The static function `create_lexicon_skeleton` will generate a lexicon "skeleton" that contains keys for each node/edge found in the provided JSON of graph information. The function itself doesn't dump this data to any file, but just returns the skeleton as a JSON object.

:::{info} Function details
:collapsible:
````{py:function} create_lexicon_skeleton(graph_json)
:canonical: pogg.lexicon.lexicon_builder.POGGLexiconUtil.create_lexicon_skeleton

```{autodoc2-docstring} pogg.lexicon.lexicon_builder.POGGLexiconUtil.create_lexicon_skeleton
```

````
:::

Imagine you have a JSON file following graph information:

```json
 {
    "nodes": {
        "cake": {
            "lexicon_key": "cake",
            "node_properties": {
                "node_type": "entity"
            }
        },
        "vanilla": {
            "lexicon_key": "vanilla",
            "node_properties": {
                "node_type": "property"
            }
        }
    },
    "edges": [
        {
            "edge_name": "flavor",
            "parent_node": "cake",
            "child_node": "vanilla",
            "lexicon_key": "flavor",
            "edge_properties": {
                "edge_type": "property"
            }
        }
    ]
}
```

This graph has two nodes (`cake` and `vanilla`) and one edge (`flavor`).

Let's call `create_lexicon_skeleton` on this graph.

In [21]:
# load the information from the JSON file
graph_json = json.load(open("graph.json"))

# create lexicon skeleton
skeleton = POGGLexiconUtil.create_lexicon_skeleton(graph_json)

pp.pprint(skeleton)

{'edge_keys': {'flavor': {'comp_fxn': ''}},
 'node_keys': {'cake': {'comp_fxn': ''}, 'vanilla': {'comp_fxn': ''}}}


The resulting skeleton has empty lexicon entries for the graph's two nodes and one edge. We can now pass this skeleton into `initialize_lexicon_directory`.

## `initialize_lexicon_directory` example

The static function `initialize_lexicon_directory` will create starter files for a lexicon provided the desired name of the lexicon and the path to store the files. If you already have a lexicon skeleton, this can be optionally passed in as the final argument.

:::{info} Function details
:collapsible:
````{py:function} initialize_lexicon_directory(lexicon_name, lexicon_directory, lexicon_skeleton)
:canonical: pogg.lexicon.lexicon_builder.POGGLexiconUtil.initialize_lexicon_directory

```{autodoc2-docstring} pogg.lexicon.lexicon_builder.POGGLexiconUtil.initialize_lexicon_directory
```

````
:::

Let's pass the skeleton created in the above example to the function.

In [22]:
# intialize directory
POGGLexiconUtil.initialize_lexicon_directory("sample", "./lexicon", skeleton)

Running this function generates the following files in the provided directory:

```
- sample_lexicon_all_entries.json
- sample_lexicon_complete_entries.json
- sample_lexicon_incomplete_entries.json
- sample_lexicon_invalid_entries.json
```

The `sample_lexicon_complete_entries.json` file and the `sample_lexicon_invalid_entries.json` files have empty lexicon dictionaries in them:

```json
{
    "node_keys": {},
    "edge_keys": {}
}
```

Whereas `sample_lexicon_incomplete_entries.json` has the skeleton we provided (and so does `sample_lexicon_all_entries.json` since this file aggregates information from every file):

```json
{
    "node_keys": {
        "cake": {
            "comp_fxn": ""
        },
        "vanilla": {
            "comp_fxn": ""
        }
    },
    "edge_keys": {
        "flavor": {
            "comp_fxn": ""
        }
    }
}
```

With our lexicon directory initialized we can work on filling out the information required for POGG's data-to-text generation algorithm.

## `update_lexicon_files` example

The static function `update_lexicon_files` reads in the lexicon files and expands entries where appropriate as well as moving those that are determined to be complete to the `lexicon_complete_entries.json` file.

:::{info} Function details
:collapsible:
````{py:function} update_lexicon_files(lexicon_directory)
:canonical: pogg.lexicon.lexicon_builder.POGGLexiconUtil.update_lexicon_files

```{autodoc2-docstring} pogg.lexicon.lexicon_builder.POGGLexiconUtil.update_lexicon_files
```

````
:::

The majority of work in creating a POGG lexicon will involve rerunning this function every time the user adds information to the `lexicon_incomplete_entries.json` file until the file is empty and everything is in the `lexicon_complete_entries_file.json`.

Using the example from above, let's add some composition functions to each entry:

```json
{
    "node_keys": {
        "cake": {
            "comp_fxn": "noun"
        },
        "vanilla": {
            "comp_fxn": "nounnn"
        }
    },
    "edge_keys": {
        "flavor": {
            "comp_fxn": "compound_noun"
        }
    }
}
```

Now we can run `update_lexicon_files` providing the lexicon directory:

In [23]:
POGGLexiconUtil.update_lexicon_files("./lexicon")

Now we have the following information in our files:

### `sample_lexicon_complete_entries.json`
```json
{
    "node_keys": {},
    "edge_keys": {}
}
```

None of our entries are complete yet, so this file remains the same.

### `sample_lexicon_incomplete_entries.json`
```json
{
    "node_keys": {
        "cake": {
            "comp_fxn": "noun",
            "predicate": "",
            "intrinsic_variable_properties": {}
        }
    },
    "edge_keys": {
        "flavor": {
            "comp_fxn": "compound_noun",
            "head_noun_sement": "",
            "non_head_noun_sement": ""
        }
    }
}
```

In the incomplete entries file we have more information that we need to fill out from the `noun` and `compound_noun` functions.

### `sample_lexicon_invalid_entries.json`
```json
{
    "node_keys": {
        "vanilla": {
            "comp_fxn": "nounnn",
            "failure_msg": "nounnn is not an existing Semantic Composition Function"
        }
    },
    "edge_keys": {}
}
```

Notice that due to the typo for the composition function entered for `"vanilla"` this entry ended up in the invalid entries file along with an error message for what the problem is.

Invalid entries can be edited directly and the next time `update_lexicon_files` is called the error messages will be removed and/or updated and the entry will be put into the appropriate file.


### One more round of edits

Let's do one more round of edits on the `sample_lexicon_incomplete_entries.json` file and the `sample_lexicon_invalid_entries.json` file and look at the results of calling `update_lexicon_files` one more time.

#### `sample_lexicon_incomplete_entries.json`
```json
{
    "node_keys": {
        "cake": {
            "comp_fxn": "noun",
            "predicate": "_cake_n_1",
            "intrinsic_variable_properties": {}
        }
    },
    "edge_keys": {
        "flavor": {
            "comp_fxn": "compound_noun",
            "head_noun_sement": "parent",
            "non_head_noun_sement": "child"
        }
    }
}
```

Here we filled out values for the parameters for each composition function.


#### `sample_lexicon_invalid_entries.json`
```json
{
    "node_keys": {
        "vanilla": {
            "comp_fxn": "noun",
            "failure_msg": "nounnn is not an existing Semantic Composition Function"
        }
    },
    "edge_keys": {}
}
```

Here we fixed the typo, leaving the error message as it will be cleared out by `update_lexicon_files`

Now let's call `update_lexicon_files` again:

In [24]:
POGGLexiconUtil.update_lexicon_files("./lexicon")

Now we have the following in each file:

#### `sample_lexicon_complete_entries.json`
```json
{
    "node_keys": {
        "cake": {
            "comp_fxn": "noun",
            "predicate": "_cake_n_1",
            "intrinsic_variable_properties": {}
        }
    },
    "edge_keys": {
        "flavor": {
            "comp_fxn": "compound_noun",
            "head_noun_sement": "parent",
            "non_head_noun_sement": "child"
        }
    }
}
```

#### `sample_lexicon_incomplete_entries.json`
```json
{
    "node_keys": {
        "vanilla": {
            "comp_fxn": "noun",
            "predicate": "",
            "intrinsic_variable_properties": {}
        }
    },
    "edge_keys": {}
}
```

#### `sample_lexicon_invalid_entries.json`
```json
{
    "node_keys": {},
    "edge_keys": {}
}
```

The complete entries have been moved to `sample_lexicon_complete_entries.json`, the fixed entry has been moved to `sample_lexicon_incomplete_entries.json`, and our `sample_lexicon_invalid_entries.json` file is now empty.

When running the data-to-text conversion algorithm any entries that are in the `complete_entries` file in the specified lexicon directory will be loaded and that will serve as the lexicon for the run.